<a href="https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:

import os

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))


Current folder: /content/flyrank-ml-internship
Files here: ['scripts', 'DATA_USE.md', '.git', 'README.md', '.github', '.gitignore', 'submission', 'notebooks', 'GUIDE.md', 'LICENSE', 'w02_ml_task_framing.ipynb', 'flyrank-ml-internship', 'CLAUDE.md', 'SETUP.md', 'w01_research_question.ipynb', 'AGENTS.md', 'skills', 'w03_data_contract.ipynb', 'docs', 'data', 'requirements.txt', 'work', 'outputs']


## 1. My rule and its reason codes
**Rule:** Prioritise content for human review when it has meaningful search visibility and signs that a refresh may be useful. The score is a transparent prioritisation heuristic, not a prediction that an item will improve.

**Reason codes:**
- `stale_visible`: content is old and has meaningful impressions.
- `position_slipping`: search position suggests an opportunity for review, provided the position measurement is valid.
- `low_evidence`: insufficient or missing data makes the recommendation uncertain; review manually before acting.

**Action:** `review_refresh_candidate`. A high score means “review first,” not “refresh automatically.”*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

root = Path("/content/flyrank-ml-internship")
input_path = root / "data/raw/content_refresh_anonymized.csv"
output_path = root / "work/outputs/baseline_action_score.csv"

df = pd.read_csv(input_path)

# Convert required signals to numeric.
for col in [
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Hand-written, interpretable signals.
visible = df["impressions_90d"].fillna(0) >= 100
stale = df["days_since_last_update"].fillna(-1) >= 180
valid_position = df["avg_position"].gt(0)
position_opportunity = valid_position & df["avg_position"].ge(8)

# Recent vs previous impressions is a descriptive signal, not a future label.
recent = df["impressions_last_30d"].fillna(0)
previous = df["impressions_prev_30d"].fillna(0)
declining = (previous >= 50) & (recent < previous * 0.8)

df["score"] = (
    stale.astype(int) * visible.astype(int) * df["impressions_90d"].fillna(0)
    + (position_opportunity & visible).astype(int) * 100
    + (declining & visible).astype(int) * 50
)

df["reason_code"] = np.select(
    [
        stale & visible,
        position_opportunity & visible,
        declining & visible,
    ],
    [
        "stale_visible",
        "position_slipping",
        "recent_impression_drop",
    ],
    default="low_evidence",
)

df["action"] = "review_refresh_candidate"
df["confidence_note"] = np.select(
    [
        stale & visible & valid_position,
        stale & visible,
        position_opportunity & visible,
        declining & visible,
    ],
    [
        "Directional: stale, visible, valid position; verify manually",
        "Directional: stale and visible; inspect context",
        "Directional: visible with weaker recorded position; verify intent",
        "Directional: recent impressions lower than prior period; verify seasonality",
    ],
    default="Low: limited evidence; inspect manually",
)

df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

df.insert(0, "rank", np.arange(1, len(df) + 1))

cols = [
    "rank", "content_id", "client_id", "score",
    "reason_code", "action", "confidence_note",
    "impressions_90d", "days_since_last_update", "avg_position",
    "impressions_last_30d", "impressions_prev_30d",
]
queue = df[cols].copy()

output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(queue))
print(queue.head(5).to_string(index=False))


Saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows: 30000
 rank           content_id         client_id  score   reason_code                   action                                              confidence_note  impressions_90d  days_since_last_update  avg_position  impressions_last_30d  impressions_prev_30d
    1 content_cf56e2e2e282 client_7f2253d7e2  61828 stale_visible review_refresh_candidate Directional: stale, visible, valid position; verify manually            61678                     194          19.7                  3864                 26791
    2 content_7368877ea310 client_7f2253d7e2  59622 stale_visible review_refresh_candidate Directional: stale, visible, valid position; verify manually            59472                     194          24.8                  3778                 20472
    3 content_1bfaa38ff26c client_7f2253d7e2  25865 stale_visible review_refresh_candidate Directional: stale, visible, valid position; verify manually       

In [ ]:

print(df.columns.tolist())


['rank', 'content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'score', 'reason_code', 'action', 'confidence_note']


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

# Review the 20 highest-ranked items from the saved queue.
from pathlib import Path
import pandas as pd

root = Path("/content/flyrank-ml-internship")
queue = pd.read_csv(root / "work/outputs/baseline_action_score.csv")
top20 = queue.head(20).copy()

def explain_reason(code):
    explanations = {
        "stale_visible": "Old update history plus meaningful historical impressions",
        "position_slipping": "Recorded average position suggests a review opportunity",
        "recent_impression_drop": "Recent impressions are lower than the previous period",
        "low_evidence": "Evidence is insufficient for a stronger recommendation",
    }
    return explanations.get(code, "Review the available signals manually")

def what_could_be_wrong(row):
    code = row["reason_code"]
    if code == "stale_visible":
        return "The page may still be performing well; age alone does not prove a refresh is needed."
    if code == "position_slipping":
        return "The query intent or SERP may explain the position; verify before changing content."
    if code == "recent_impression_drop":
        return "Seasonality, tracking changes, or demand shifts may explain the drop."
    return "Missing or weak evidence may make this recommendation unhelpful."

top20["why_here"] = top20["reason_code"].apply(explain_reason)
top20["what_would_make_it_wrong"] = top20.apply(what_could_be_wrong, axis=1)

for _, row in top20.iterrows():
    print(
        f"Rank {int(row['rank'])} | {row['content_id']}\n"
        f"Action: {row['action']}\n"
        f"Reason: {row['reason_code']} — {row['why_here']}\n"
        f"Confidence: {row['confidence_note']}\n"
        f"What could make it wrong: {row['what_would_make_it_wrong']}\n"
    )

print("Top-20 rows reviewed:", len(top20))



Rank 1 | content_cf56e2e2e282
Action: review_refresh_candidate
Reason: stale_visible — Old update history plus meaningful historical impressions
Confidence: Directional: stale, visible, valid position; verify manually
What could make it wrong: The page may still be performing well; age alone does not prove a refresh is needed.

Rank 2 | content_7368877ea310
Action: review_refresh_candidate
Reason: stale_visible — Old update history plus meaningful historical impressions
Confidence: Directional: stale, visible, valid position; verify manually
What could make it wrong: The page may still be performing well; age alone does not prove a refresh is needed.

Rank 3 | content_1bfaa38ff26c
Action: review_refresh_candidate
Reason: stale_visible — Old update history plus meaningful historical impressions
Confidence: Directional: stale, visible, valid position; verify manually
What could make it wrong: The page may still be performing well; age alone does not prove a refresh is needed.

Rank 4 | c

**Weak picks:** Some high-ranked items may be false positives. A page can have many historical impressions but no longer need a refresh; an old update date alone does not establish a content problem. Average position can also be affected by query mix and search intent. The queue is for human review, not automatic action.

**Leakage check:** The score uses historical impressions, update age, recorded average position, and the previous-versus-recent 30-day impression comparison. It does not use `trend_direction` or `trend_pct`, which are label-derived, and it does not use product flags or future-window outcomes. The recommendations are directional and need manual validation.## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Final self-check**

- The rule is a transparent, hand-written prioritisation heuristic.
- The ranked queue contains 30,000 rows and is regenerated by the notebook.
- The top 20 have an action, reason code, confidence note, and a note explaining what could make each recommendation wrong.
- The score does not use `trend_direction`, `trend_pct`, product flags, or future-window outcomes.
- The queue is for decision-support and human review, not an automatic refresh decision.
- Results are directional; weak picks and limitations are acknowledged.## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:

!git clone https://github.com/arshiyanaaz411/flyrank-ml-internship.git



fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [ ]:

%cd /content/flyrank-ml-internship

import os

print("Root files:", os.listdir("."))
print("\nSkills folder exists:", os.path.exists("skills"))
print("README exists:", os.path.exists("skills/README.md"))
print("Target notebook exists:", os.path.exists("work/notebooks/w04_baseline_score.ipynb"))


/content/flyrank-ml-internship
Root files: ['scripts', 'DATA_USE.md', '.git', 'README.md', '.github', '.gitignore', 'submission', 'notebooks', 'GUIDE.md', 'LICENSE', 'w02_ml_task_framing.ipynb', 'flyrank-ml-internship', 'CLAUDE.md', 'SETUP.md', 'w01_research_question.ipynb', 'AGENTS.md', 'skills', 'w03_data_contract.ipynb', 'docs', 'data', 'requirements.txt', 'work', 'outputs']

Skills folder exists: True
README exists: True
Target notebook exists: True


In [ ]:

%cd /content/flyrank-ml-internship

with open("skills/README.md", "r", encoding="utf-8") as f:
    print(f.read())


/content/flyrank-ml-internship
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/fly

In [ ]:

%cd /content/flyrank-ml-internship

for path in [
    "skills/building-baselines/SKILL.md",
    "skills/flyrank/flyrank-data/SKILL.md"
]:
    print("\n" + "=" * 70)
    print(path)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        print(f.read())


/content/flyrank-ml-internship

skills/building-baselines/SKILL.md
---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]    